In [ ]:
!uname -a && cat /etc/*release
!pwd
!ls -la

In [1]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

// --- KERNEL NAIVE ---
// Nessuna ottimizzazione, nessuna memoria condivisa.
// Ogni thread legge direttamente dalla lentissima Memoria Globale.
__global__ void matMulNaive(const double *a, const double *b, double *c, int n) {
    // I cicli "i" e "j" scompaiono: le coordinate sono calcolate tramite gli indici dei thread
    int row = blockIdx.y * blockDim.y + threadIdx.y; // Sostituisce la 'i'
    int col = blockIdx.x * blockDim.x + threadIdx.x; // Sostituisce la 'j'

    // Controllo dei bordi: ci assicuriamo che il thread sia dentro la matrice
    if (row < n && col < n) {
        double sum = 0.0;

        // Questo è l'unico ciclo che sopravvive (corrisponde al ciclo 'k' originale)
        for (int k = 0; k < n; ++k) {
            // Lettura pesantissima direttamente dalla memoria globale (VRAM)
            sum += a[row * n + k] * b[k * n + col];
        }

        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Error: missing n argument.\\n");
        fprintf(stderr, "Usage: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    if (n <= 0) {
        fprintf(stderr, "Error: You forgot to provide n!\\n");
        return 1;
    }

    // Calcoliamo i byte necessari (usando double: 8 byte per elemento)
    size_t bytes = (size_t)n * n * sizeof(double);

    // Allocazione Host (CPU)
    // NOTA: In CUDA si preferisce sempre "appiattire" le matrici in array 1D
    double *h_a = (double*)malloc(bytes);
    double *h_b = (double*)malloc(bytes);
    double *h_c = (double*)malloc(bytes);

    if (!h_a || !h_b || !h_c) {
        fprintf(stderr, "Errore di allocazione RAM Host!\\n");
        return 1;
    }

    // Inizializzazione
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0;
            h_b[i * n + j] = 3.0;
            h_c[i * n + j] = 0.0;
        }
    }

    // Allocazione Device (GPU)
    double *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Trasferimento dati HtoD
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Configurazione della Griglia di esecuzione (Blocchi 32x32 = 1024 thread)
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation (CUDA Naive - FP64)...\\n");
    clock_t start = clock();

    // Lancio del kernel
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // La CPU deve aspettare che la GPU finisca prima di fermare il cronometro
    cudaDeviceSynchronize();
    clock_t end = clock();

    double duration = (double)(end - start) / CLOCKS_PER_SEC;
    printf("Execution Time: %.4f seconds\\n", duration);

    // Trasferimento risultati DtoH
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Salvataggio dei risultati (con salvaguardia per n piccoli)
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen");
        return 1;
    }

    fprintf(f, "%d\\n\\n", n);
    int sample_size = (n < 1000) ? n : 1000;
    for (int i = 0; i < sample_size; i++) {
        for (int j = 0; j < sample_size; j++) {
            fprintf(f, "%.0f ", h_c[i * n + j]);
        }
        fprintf(f, "\\n");
    }

    fclose(f);

    // Pulizia finale
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o cuda_matrixmult_naive
!nvprof ./cuda_matrixmult_naive 5000

==8750== NVPROF is profiling process 8750, command: ./cuda_matrixmult_naive 5000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 1.2601 seconds
==8750== Profiling application: ./cuda_matrixmult_naive 5000
==8750== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   90.87%  1.25994s         1  1.25994s  1.25994s  1.25994s  matMulNaive(double const *, double const *, double*, int)
                    6.06%  83.984ms         2  41.992ms  41.688ms  42.296ms  [CUDA memcpy HtoD]
                    3.07%  42.546ms         1  42.546ms  42.546ms  42.546ms  [CUDA memcpy DtoH]
      API calls:   78.52%  1.25995s         1  1.25995s  1.25995s  1.25995s  cudaDeviceSynchronize
                   12.98%  208.29ms         3  69.430ms  80.773us  208.09ms  cudaMalloc
                    7.98%  128.04ms         3  42.679ms  42.518ms  42.954ms  cudaMemcpy
                    0.22%  3.5767ms         3  1.1922ms  313.08us  1

In [3]:
!nvprof ./cuda_matrixmult_naive 10000

==8782== NVPROF is profiling process 8782, command: ./cuda_matrixmult_naive 10000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 10.1659 seconds
==8782== Profiling application: ./cuda_matrixmult_naive 10000
==8782== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   95.09%  10.2202s         1  10.2202s  10.2202s  10.2202s  matMulNaive(double const *, double const *, double*, int)
                    3.14%  337.71ms         2  168.86ms  168.19ms  169.52ms  [CUDA memcpy HtoD]
                    1.77%  189.86ms         1  189.86ms  189.86ms  189.86ms  [CUDA memcpy DtoH]
      API calls:   93.58%  10.2203s         1  10.2203s  10.2203s  10.2203s  cudaDeviceSynchronize
                    4.84%  528.45ms         3  176.15ms  168.41ms  190.32ms  cudaMemcpy
                    1.51%  165.35ms         3  55.117ms  107.57us  165.12ms  cudaMalloc
                    0.05%  5.1215ms         3  1.7072ms  565.29us

In [4]:
!nvprof ./cuda_matrixmult_naive 15000

==8852== NVPROF is profiling process 8852, command: ./cuda_matrixmult_naive 15000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 42.3556 seconds
==8852== Profiling application: ./cuda_matrixmult_naive 15000
==8852== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.38%  42.5425s         1  42.5425s  42.5425s  42.5425s  matMulNaive(double const *, double const *, double*, int)
                    1.76%  768.05ms         2  384.03ms  382.52ms  385.53ms  [CUDA memcpy HtoD]
                    0.86%  376.60ms         1  376.60ms  376.60ms  376.60ms  [CUDA memcpy DtoH]
      API calls:   96.96%  42.5425s         1  42.5425s  42.5425s  42.5425s  cudaDeviceSynchronize
                    2.61%  1.14555s         3  381.85ms  377.00ms  385.79ms  cudaMemcpy
                    0.41%  178.90ms         3  59.632ms  145.94us  178.60ms  cudaMalloc
                    0.02%  8.2234ms         3  2.7411ms  2.4022ms

In [5]:
!nvprof ./cuda_matrixmult_naive 20000

==9061== NVPROF is profiling process 9061, command: ./cuda_matrixmult_naive 20000
Starting the computation (CUDA Naive - FP64)...
Execution Time: 97.8141 seconds
==9061== Profiling application: ./cuda_matrixmult_naive 20000
==9061== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.96%  98.7881s         1  98.7881s  98.7881s  98.7881s  matMulNaive(double const *, double const *, double*, int)
                    1.35%  1.35715s         2  678.58ms  674.20ms  682.95ms  [CUDA memcpy HtoD]
                    0.69%  697.32ms         1  697.32ms  697.32ms  697.32ms  [CUDA memcpy DtoH]
      API calls:   97.77%  98.7882s         1  98.7882s  98.7882s  98.7882s  cudaDeviceSynchronize
                    2.03%  2.05542s         3  685.14ms  674.42ms  697.84ms  cudaMemcpy
                    0.19%  191.42ms         3  63.807ms  218.63us  190.95ms  cudaMalloc
                    0.01%  8.3520ms         3  2.7840ms  1.8274ms